In [1]:
import pandas as pd

In [2]:
data = {
    "Transaction_ID": [
        "TXN001", "TXN002", "TXN002", "TXN003",
        "TXN004", "TXN005", "TXN006", "TXN007"
    ],

    "Customer_ID": [
        "C101", "C102", "C102", "C103",
        None, "C105", "C106", "C107"
    ],

    "Amount": [
        "1250.50", "5000.00", "5000.00", "750.25",
        "3000.00", "12000.00", "450.00", "900.00"
    ],

    "Status": [
        " SUCCESS ", "success", "success", "FAILED",
        "Success ", None, " failed ", "SUCCESS"
    ],

    "Channel": [
        "UPI", " card ", " card ", "NETBANKING",
        "UPI", "CARD", None, " upi "
    ]
}

transactions = pd.DataFrame(data)

### Data Inspection

You first inspect the incoming dataset and its quality

In [3]:
transactions.shape

(8, 5)

In [4]:
transactions.head()

,Transaction_ID,Customer_ID,Amount,Status,Channel
0,TXN001,C101,1250.50,SUCCESS,UPI
1,TXN002,C102,5000.00,success,card
2,TXN002,C102,5000.00,success,card
3,TXN003,C103,750.25,FAILED,NETBANKING
4,TXN004,NaN,3000.00,Success,UPI


In [5]:
transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Transaction_ID  8 non-null      str  
 1   Customer_ID     7 non-null      str  
 2   Amount          8 non-null      str  
 3   Status          7 non-null      str  
 4   Channel         7 non-null      str  
dtypes: str(5)
memory usage: 452.0 bytes


In [6]:
transactions.isna().sum()

Transaction_ID    0
Customer_ID       1
Amount            0
Status            1
Channel           1
dtype: int64

### Cleaning

Transaction_ID is renamed to transaction_id and Customer_ID to customer_id

In [7]:
transactions.rename(columns={
    "Transaction_ID": "transaction_id",
    "Customer_ID": "customer_id"
}, inplace=True)

In [8]:
transactions.columns

Index(['transaction_id', 'customer_id', 'Amount', 'Status', 'Channel'], dtype='str')

Duplicate transactions based on transaction_id are removed

In [9]:
transactions[transactions.duplicated(subset=['transaction_id'])]

,transaction_id,customer_id,Amount,Status,Channel
2,TXN002,C102,5000.00,success,card


Verifying if data is duplicated on all the columns for duplicate transactions

In [10]:
transactions[transactions['transaction_id'] == 'TXN002']

,transaction_id,customer_id,Amount,Status,Channel
1,TXN002,C102,5000.00,success,card
2,TXN002,C102,5000.00,success,card


In [11]:
transactions.drop_duplicates(subset=['transaction_id'], inplace=True)

In [12]:
transactions

,transaction_id,customer_id,Amount,Status,Channel
0,TXN001,C101,1250.50,SUCCESS,UPI
1,TXN002,C102,5000.00,success,card
3,TXN003,C103,750.25,FAILED,NETBANKING
4,TXN004,NaN,3000.00,Success,UPI
5,TXN005,C105,12000.00,NaN,CARD
6,TXN006,C106,450.00,failed,NaN
7,TXN007,C107,900.00,SUCCESS,upi


Transactions without a customer_id are considered invalid and removed

In [13]:
transactions[transactions['customer_id'].isna()]

,transaction_id,customer_id,Amount,Status,Channel
4,TXN004,NaN,3000.00,Success,UPI


In [14]:
transactions.dropna(subset=['customer_id'], inplace=True)

In [15]:
transactions

,transaction_id,customer_id,Amount,Status,Channel
0,TXN001,C101,1250.50,SUCCESS,UPI
1,TXN002,C102,5000.00,success,card
3,TXN003,C103,750.25,FAILED,NETBANKING
5,TXN005,C105,12000.00,NaN,CARD
6,TXN006,C106,450.00,failed,NaN
7,TXN007,C107,900.00,SUCCESS,upi


Amount must be a numeric/float column

In [16]:
transactions.dtypes

transaction_id    str
customer_id       str
Amount            str
Status            str
Channel           str
dtype: object

In [17]:
transactions['Amount'] = transactions['Amount'].astype('float64')

In [18]:
transactions.dtypes

transaction_id        str
customer_id           str
Amount            float64
Status                str
Channel               str
dtype: object

Status must have consistent uppercase values with surrounding spaces removed.

In [19]:
transactions['Status']

0     SUCCESS 
1      success
3       FAILED
5          NaN
6      failed 
7      SUCCESS
Name: Status, dtype: str

In [20]:
transactions['Status'] = transactions['Status'].str.strip().str.upper()

In [21]:
transactions['Status']

0    SUCCESS
1    SUCCESS
3     FAILED
5        NaN
6     FAILED
7    SUCCESS
Name: Status, dtype: str

Missing Status should become "UNKNOWN".

In [22]:
transactions['Status'] = transactions['Status'].fillna(value="UNKNOWN")

In [23]:
transactions['Status']

0    SUCCESS
1    SUCCESS
3     FAILED
5    UNKNOWN
6     FAILED
7    SUCCESS
Name: Status, dtype: str

Channel must also be standardized by removing surrounding spaces and converting values to uppercase.

In [24]:
transactions['Channel']

0           UPI
1         card 
3    NETBANKING
5          CARD
6           NaN
7          upi 
Name: Channel, dtype: str

In [25]:
transactions['Channel'] = transactions['Channel'].str.strip().str.upper()

In [26]:
transactions['Channel']

0           UPI
1          CARD
3    NETBANKING
5          CARD
6           NaN
7           UPI
Name: Channel, dtype: str

Missing Channel should become "UNKNOWN".

In [27]:
transactions['Channel'] = transactions['Channel'].fillna(value="UNKNOWN")

In [28]:
transactions

,transaction_id,customer_id,Amount,Status,Channel
0,TXN001,C101,1250.50,SUCCESS,UPI
1,TXN002,C102,5000.00,SUCCESS,CARD
3,TXN003,C103,750.25,FAILED,NETBANKING
5,TXN005,C105,12000.00,UNKNOWN,CARD
6,TXN006,C106,450.00,FAILED,UNKNOWN
7,TXN007,C107,900.00,SUCCESS,UPI


In [29]:
transactions.info()

<class 'pandas.DataFrame'>
Index: 6 entries, 0 to 7
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   transaction_id  6 non-null      str    
 1   customer_id     6 non-null      str    
 2   Amount          6 non-null      float64
 3   Status          6 non-null      str    
 4   Channel         6 non-null      str    
dtypes: float64(1), str(4)
memory usage: 288.0 bytes


In [31]:
transactions.count()

transaction_id    6
customer_id       6
Amount            6
Status            6
Channel           6
dtype: int64

In [35]:
transactions[transactions['transaction_id'].duplicated()]

,transaction_id,customer_id,Amount,Status,Channel


In [40]:
transactions.isna().value_counts()

transaction_id  customer_id  Amount  Status  Channel
False           False        False   False   False      6
Name: count, dtype: int64

In [30]:
transactions['Amount'].sum()

np.float64(20350.75)

Number of input records = 8 \
Number of output records = 6 \
Number of duplicate transaction_ids remaining = 0 \
Number of nulls remaining = 0 \
Final column data types = Amount - float64, rest all str
Total transaction amount in the cleaned dataset = 20350.75